# QuickDev · demo

QuickDev analiza el feedback de un playtest cerrado (antes del lanzamiento, cuando todavía no hay
reseñas de Steam) y entrega un reporte JSON agregado: problemas priorizados, citas de evidencia,
comentarios descartados con su motivo y un flag de revisión humana.

> **El modelo interpreta lenguaje. El código verifica hechos.**

**Este notebook no contiene lógica.** Todo lo que ejecuta vive en el paquete `quickdev/`; aquí solo
se importa y se muestra. Antes era el producto entero y eso causó que el validador existiera dos
veces y divergiera ([ADR-0010](docs/adr/0010-notebook-como-demo-delgada.md)). El razonamiento que
vivía en sus celdas está en [`docs/historia/`](docs/historia/README.md).

Corre sin API key y sin red: la respuesta del modelo es la que se grabó al medir el prompt `after`
(`evals/resultados/after/crudo.json`). Todo lo que pasa después de la llamada es el código de verdad.
Requiere `pip install -e ".[dev]"` y abrir el notebook desde la raíz del repo. Lo mismo, en terminal:
`quickdev demo`.

In [ ]:
# 1. El lote de entrada: 14 comentarios reales del playtest de la build v0.8.2
from quickdev.adapters import FakeLlm
from quickdev.cli import CRUDO_DEMO, LOTE_DEMO, construir_analisis, leer_lote

lote = leer_lote(LOTE_DEMO)
print(f"build {lote.build} · {lote.total} comentarios\n")
for i, c in enumerate(lote.comentarios):
    print(f"{i:>2} ({c.fuente}) {c.texto}")

In [ ]:
# 2. El pipeline completo: prompt -> modelo -> parseo -> validacion -> reparacion
analisis = construir_analisis(
    FakeLlm.from_crudo(CRUDO_DEMO), prompt_version="after", rules_version="after"
)
resultado = analisis.execute(lote)

reporte = resultado.report
print(reporte.resumen_general, "\n")
for p in reporte.problemas_detectados:
    print(f"[{p.prioridad:5}] {p.categoria:<11} x{p.frecuencia}  {p.descripcion}")
print("\nDescartados (nada desaparece en silencio):")
for d in reporte.comentarios_descartados:
    print(f"  ({d.motivo}) {d.texto}")

In [ ]:
# 3. Lo que decide el codigo: hallazgos del validador, revision humana y la traza de la ejecucion
print("hallazgos:", [h.rule_id for h in resultado.findings] or "ninguno")
print("requiere_revision_humana:", reporte.requiere_revision_humana, "\n")
for paso in resultado.trace.steps:
    print(f"{paso.name:<16} {paso.decision}")

In [ ]:
# 4. Romperlo a proposito: si el modelo inventara el conteo, el codigo lo atrapa y lo corrige
from quickdev.domain.rules.registry import rule_set
from quickdev.domain.validation import RepairPolicy, Validator

inventado = reporte.model_copy(
    update={"total_comentarios_analizados": 214, "requiere_revision_humana": False}
)
hallazgos = Validator(rule_set("after")).validate(inventado, lote)
for h in hallazgos:
    print(f"{h.rule_id}: {h.message}")

corregido = RepairPolicy().repair(inventado, lote, hallazgos)
print("\ntotal corregido:", corregido.total_comentarios_analizados)
print("requiere_revision_humana:", corregido.requiere_revision_humana)